<a href="https://colab.research.google.com/github/krasnovaov81-coder/Case_DZ_Netology/blob/main/%D0%9E%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D1%87%D0%B8%D0%BA_CSV_%D1%81_%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%BE%D0%B9_%D1%83%D1%81%D1%82%D0%BE%D0%B9%D1%87%D0%B8%D0%B2%D0%BE%D1%81%D1%82%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое задание. Обработчик CSV с проверкой устойчивости

## Описание задания

Вы — аналитик в digital-агентстве. Клиент предоставил выгрузку данных о рекламных кампаниях за последний месяц. Данные пришли из разных источников, поэтому в файле есть пропуски, ошибки и некорректные значения.

Ваша задача — создать надёжный скрипт обработки CSV, который:
- проверит структуру и качество данных;
- рассчитает ключевые метрики эффективности;
- сформирует отчёт для клиента.

## Цели задания
- Научиться читать и записывать CSV-файлы с помощью модуля `csv`
- Освоить проверку структуры данных и валидацию
- Научиться обрабатывать некорректные данные через `try...except`
- Освоить расчёт производных показателей
- Понять принципы построения устойчивого пайплайна обработки данных

## Исходные данные

Файл `campaigns_raw.csv` содержит данные о рекламных кампаниях:
- `campaign_id` — идентификатор кампании
- `campaign_name` — название кампании
- `platform` — платформа (Яндекс, Google, VK)
- `impressions` — количество показов
- `clicks` — количество кликов
- `cost` — расходы в рублях
- `conversions` — количество конверсий
- `revenue` — выручка в рублях

### Шаг 0. Подготовка тестового файла

Сначала создадим тестовый файл с данными, которые будем обрабатывать.
В реальном задании этот файл уже был бы предоставлен.

In [ ]:
import csv
from pathlib import Path

# Определяем путь к файлу
data_dir = Path("/content")
source_file = data_dir / "campaigns_raw.csv"

# Исходные данные с разными типами ошибок
raw_campaigns = [
    {"campaign_id": "101", "campaign_name": "Летняя распродажа", "platform": "Яндекс", "impressions": "15000", "clicks": "450", "cost": "12000", "conversions": "25", "revenue": "85000"},
    {"campaign_id": "102", "campaign_name": "Брендовый трафик", "platform": "Google", "impressions": "25000", "clicks": "800", "cost": "25000", "conversions": "12", "revenue": "45000"},
    {"campaign_id": "103", "campaign_name": "Товарная кампания", "platform": "Яндекс", "impressions": "8000", "clicks": "200", "cost": "", "conversions": "8", "revenue": "32000"},
    {"campaign_id": "104", "campaign_name": "Ретаргетинг", "platform": "VK", "impressions": "12000", "clicks": "н/д", "cost": "5000", "conversions": "15", "revenue": "28000"},
    {"campaign_id": "105", "campaign_name": "Поиск клиентов", "platform": "Google", "impressions": "30000", "clicks": "900", "cost": "35000", "conversions": "", "revenue": ""},
    {"campaign_id": "106", "campaign_name": "Новая кампания", "platform": "Яндекс", "impressions": "5000", "clicks": "150", "cost": "8000", "conversions": "10", "revenue": "42000"},
    {"campaign_id": "107", "campaign_name": "Тестовая", "platform": "VK", "impressions": "ноль", "clicks": "50", "cost": "3000", "conversions": "3", "revenue": "15000"},
    {"campaign_id": "108", "campaign_name": "Мобильный трафик", "platform": "Google", "impressions": "18000", "clicks": "600", "cost": "18000", "conversions": "20", "revenue": "72000"},
    {"campaign_id": "109", "campaign_name": "Видео реклама", "platform": "VK", "impressions": "22000", "clicks": "700", "cost": "15000", "conversions": "18", "revenue": "55000"},
    {"campaign_id": "110", "campaign_name": "Динамический ретаргетинг", "platform": "Яндекс", "impressions": "10000", "clicks": "300", "cost": "9000", "conversions": "12", "revenue": "38000"},
]

# Создаём файл
with open(source_file, mode="w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=raw_campaigns[0].keys())
    writer.writeheader()
    writer.writerows(raw_campaigns)

print(f"Тестовый файл создан: {source_file}")
print(f"Всего кампаний в исходных данных: {len(raw_campaigns)}")

Тестовый файл создан: /content/campaigns_raw.csv
Всего кампаний в исходных данных: 10


In [ ]:
# Ваше решение:
import csv
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# --- Задание 1. Проверка структуры файла ---
def check_file_structure(file_path: Path, required_columns: List[str]) -> bool:
    """
    Проверяет существование файла и наличие обязательных столбцов.
    Возвращает True, если структура корректна, иначе False.
    """
    if not file_path.exists():
        print(f"Ошибка: Файл не найден по пути {file_path}")
        return False

    try:
        with open(file_path, mode="r", encoding="utf-8") as f:
            reader = csv.reader(f)
            headers = next(reader, None)
            if headers is None:
                print("Ошибка: Файл пуст или не содержит заголовков.")
                return False

            missing_cols = [col for col in required_columns if col not in headers]
            if missing_cols:
                print(f"Ошибка: В файле отсутствуют обязательные столбцы: {', '.join(missing_cols)}")
                return False
    except Exception as e:
        print(f"Ошибка при чтении файла: {e}")
        return False

    return True

# --- Задание 2. Чтение и валидация данных ---
def read_and_validate(file_path: Path, numeric_columns: List[str]) -> Tuple[List[Dict], List[Dict]]:
    """
    Читает CSV-файл и проверяет строки на наличие ошибок в числовых полях.
    Возвращает кортеж: (список строк, список ошибок).
    """
    data = []
    errors = []

    with open(file_path, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row_number, row in enumerate(reader, start=2): # start=2, так как 1-я строка — заголовок
            for col in numeric_columns:
                value = row.get(col, "")
                if value is None or (isinstance(value, str) and value.strip() == ""):
                    errors.append({
                        "row_number": row_number,
                        "column": col,
                        "value": value,
                        "error_type": "empty"
                    })
                else:
                    try:
                        # Проверяем возможность преобразования без приведения типов
                        float(value.replace(",", ".")) # Учитываем запятую как разделитель
                    except (ValueError, AttributeError):
                        errors.append({
                            "row_number": row_number,
                            "column": col,
                            "value": value,
                            "error_type": "invalid_format"
                        })
            data.append(row)

    return data, errors

# --- Задание 3. Очистка данных с обработкой исключений ---
def clean_data(raw_data: List[Dict]) -> List[Dict]:
    """
    Очищает данные: приводит типы, пропускает строки с критическими ошибками.
    Возвращает список очищенных словарей.
    """
    cleaned_data = []

    for row in raw_data:
        try:
            # Критические поля (если ошибка — пропускаем строку)
            impressions = int(row["impressions"])
            clicks = int(row["clicks"])
            cost = float(row["cost"])

            # Некритические поля (пустые -> 0)
            conversions = int(row["conversions"]) if row["conversions"] not in ("", None) else 0
            revenue = float(row["revenue"]) if row["revenue"] not in ("", None) else 0.0

            cleaned_row = {
                "campaign_id": row["campaign_id"],
                "campaign_name": row["campaign_name"],
                "platform": row["platform"],
                "impressions": impressions,
                "clicks": clicks,
                "cost": cost,
                "conversions": conversions,
                "revenue": revenue
            }
            cleaned_data.append(cleaned_row)
        except (ValueError, TypeError):
            # Пропускаем строки с ошибками в критических числовых полях
            continue

    return cleaned_data

# --- Задание 4. Расчёт метрик эффективности ---
def calculate_metrics(data: List[Dict]) -> List[Dict]:
    """
    Добавляет к данным рассчитанные метрики CTR, CPC, CPA, ROI.
    Возвращает новый список словарей.
    """
    result = []

    for row in data:
        impressions = row["impressions"]
        clicks = row["clicks"]
        conversions = row["conversions"]
        cost = row["cost"]
        revenue = row["revenue"]

        # CTR
        ctr = round((clicks / impressions) * 100, 2) if impressions > 0 else None

        # CPC
        cpc = round(cost / clicks, 2) if clicks > 0 else None

        # CPA
        cpa = round(cost / conversions, 2) if conversions > 0 else None

        # ROI
        roi = round(((revenue - cost) / cost) * 100, 2) if cost > 0 else None

        new_row = row.copy()
        new_row.update({
            "ctr": ctr,
            "cpc": cpc,
            "cpa": cpa,
            "roi": roi
        })
        result.append(new_row)

    return result

# --- Задание 5. Фильтрация и агрегация ---
def filter_campaigns_by_roi(data: List[Dict], min_roi: float) -> List[Dict]:
    """Возвращает список кампаний с ROI >= min_roi."""
    return [row for row in data if row["roi"] is not None and row["roi"] >= min_roi]

def aggregate_by_platform(data: List[Dict]) -> Dict[str, Dict]:
    """
    Агрегирует данные по платформам.
    Возвращает словарь {platform: {total_cost, total_revenue, ...}}.
    """
    agg = {}

    for row in data:
        platform = row["platform"]
        if platform not in agg:
            agg[platform] = {
                "total_cost": 0.0,
                "total_revenue": 0.0,
                "total_conversions": 0,
                "roi_values": [],
                "campaigns_count": 0
            }

        agg[platform]["total_cost"] += row["cost"]
        agg[platform]["total_revenue"] += row["revenue"]
        agg[platform]["total_conversions"] += row["conversions"]

        if row["roi"] is not None:
            agg[platform]["roi_values"].append(row["roi"])

        agg[platform]["campaigns_count"] += 1

    # Вычисляем средний ROI для каждой платформы
    for platform_info in agg.values():
        roi_values = platform_info.pop("roi_values")
        platform_info["avg_roi"] = round(sum(roi_values) / len(roi_values), 2) if roi_values else None

    return agg

# --- Задание 6. Сохранение отчёта в CSV ---
def save_report(data: List[Dict], output_path: Path) -> bool:
    """
    Сохраняет данные в CSV-файл.
    Возвращает True при успехе, False при ошибке.
    """
    fieldnames_order = [
        "campaign_id", "campaign_name", "platform",
        "impressions", "clicks", "cost", "conversions", "revenue",
        "ctr", "cpc", "cpa", "roi"
    ]

    try:
        with open(output_path, mode="w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames_order)
            writer.writeheader()
            for row in data:
                # Приводим None к пустой строке для красивого вывода в CSV
                safe_row = {k: "" if v is None else v for k, v in row.items()}
                writer.writerow(safe_row)
        print(f"Отчёт успешно сохранён: {output_path}")
        return True
    except Exception as e:
        print(f"Ошибка при сохранении отчёта: {e}")
        return False

# --- Задание 7. Итоговый пайплайн ---
def run_pipeline(input_file: Path, output_file: Path, min_roi_threshold: float) -> bool:
    """
    Главный пайплайн обработки данных.
    Возвращает True при успешном выполнении.
    """
    required_columns = ["campaign_id", "campaign_name", "platform", "impressions", "clicks", "cost", "conversions", "revenue"]

    print("Шаг 1: Проверка структуры файла...")
    if not check_file_structure(input_file, required_columns):
        return False

    print("Шаг 2: Чтение и первичная валидация данных...")
    raw_data, validation_errors = read_and_validate(input_file, ["impressions", "clicks", "cost", "conversions", "revenue"])

    if validation_errors:
        print(f"Найдено {len(validation_errors)} ошибок валидации:")
        for err in validation_errors[:5]: # Покажем первые 5 для примера
            print(f"Строка {err['row_number']}, колонка '{err['column']}': {err['error_type']} (значение: {err['value']})")

    print("Шаг 3: Очистка данных...")
    cleaned_data = clean_data(raw_data)

    if not cleaned_data:
        print("Ошибка: После очистки не осталось валидных данных.")
        return False

    print("Шаг 4: Расчёт метрик эффективности...")
    data_with_metrics = calculate_metrics(cleaned_data)

    print("Шаг 5: Фильтрация и агрегация...")
    effective_campaigns = filter_campaigns_by_roi(data_with_metrics, min_roi_threshold)
    aggregation_report = aggregate_by_platform(data_with_metrics)

    print("\nСводный отчёт по платформам:")
    for platform, stats in aggregation_report.items():
        print(f"- {platform}: Затраты {stats['total_cost']:.2f} ₽ | Выручка {stats['total_revenue']:.2f} ₽ | Конверсии {stats['total_conversions']} | Кампаний {stats['campaigns_count']} | Средний ROI {stats['avg_roi']}%")

    print(f"\nНайдено кампаний с ROI >= {min_roi_threshold}%: {len(effective_campaigns)}")

    print("Шаг 6: Сохранение итогового отчёта...")
    save_success = save_report(data_with_metrics, output_file)

    return save_success

# --- Запуск пайплайна ---
if __name__ == "__main__":
    data_dir = Path("/content")

    input_file_path = data_dir / "campaigns_raw.csv"
    output_file_path = data_dir / "campaigns_report.csv"

    roi_threshold = 150.0

    pipeline_result = run_pipeline(input_file_path, output_file_path, roi_threshold)
    print("\nИтог выполнения пайплайна:", "УСПЕШНО" if pipeline_result else "СБОЙ")

Шаг 1: Проверка структуры файла...
Шаг 2: Чтение и первичная валидация данных...
Найдено 5 ошибок валидации:
Строка 4, колонка 'cost': empty (значение: )
Строка 5, колонка 'clicks': invalid_format (значение: н/д)
Строка 6, колонка 'conversions': empty (значение: )
Строка 6, колонка 'revenue': empty (значение: )
Строка 8, колонка 'impressions': invalid_format (значение: ноль)
Шаг 3: Очистка данных...
Шаг 4: Расчёт метрик эффективности...
Шаг 5: Фильтрация и агрегация...

Сводный отчёт по платформам:
- Яндекс: Затраты 29000.00 ₽ | Выручка 165000.00 ₽ | Конверсии 47 | Кампаний 3 | Средний ROI 451.85%
- Google: Затраты 78000.00 ₽ | Выручка 117000.00 ₽ | Конверсии 32 | Кампаний 3 | Средний ROI 93.33%
- VK: Затраты 15000.00 ₽ | Выручка 55000.00 ₽ | Конверсии 18 | Кампаний 1 | Средний ROI 266.67%

Найдено кампаний с ROI >= 150.0%: 5
Шаг 6: Сохранение итогового отчёта...
Отчёт успешно сохранён: /content/campaigns_report.csv

Итог выполнения пайплайна: УСПЕШНО
